---
title: "Implementing the Metadata Transformation"
---

This section illustrates the "transform" process, which in this case
also demonstrates the implementation of the Metadata Application Profile (MAP). (See [](/part03/apis-04-map-design.ipynb) for a discussion of MAP creation.)
There are many possible data transformation pathways, and differences may depend on different approaches to system architecture, overall metadata schemes, and the technical resources at your disposal, among other factors.
Keep in mind that this demonstration is based on
the unique requirements of your digital collection and aims to
adopt open-source values by illustrating a process that
converts metadata into an open, non-proprietary,
text-based format: a CSV file.

As throughout, the general process here follows the generalized "Extract - Transform - Load" process, which is frequently the abstract model for pulling data from one system, transporting, cleaning, and outputting to another system, which remains the overall workflow.

:::{important} Learning objectives

After completing the assignment associated with this notebook, you should: 

- Use a script (i.e., a small Python program) to work with data supplied by an API in JSON format to manage and transform useful parts of that data into a CSV format.
- Create ingest-ready collection metadata that conforms to Dublin Core and other digital collection metadata standards, which can be used to load content into another site (in this case, a new collection platform, like Omeka S or CollectionBuilder).
:::

# Introduction

The high-level steps comprise the basic elements of a workflow to **transform** the descriptive metadata gathered from an API. This step also assumes you have already developed a plan for transforming your data, which is documented in a [Metadata Application Profile, or MAP](/part03/apis-04-map-design.ipynb).
The basic steps include:

1. Develop your transformation script according to the metadata crosswalk defined in the MAP, then test the script with a subset of the metadata (i.e., one or two records). The section below demonstrates this with one record, but a production environment will have many additional challenges that may arise, so this step would be extended.
2. Transform the data you've gathered into the desired form. The example below illustartes a process that pulls data from a series of JSON-formatted item records into a single CSV file. The goal is to create a CSV that accurately represents the set of items in an Omeka S site using the CSV Import [module](/part03/servers-omeka-s-add-modules.md). 

:::{note} The Code Below Demonstrates the Process
The code outlined below suggests how all of these data elements may be extracted and transformed, but it does not necessarily output all of the elements necessary to complete the transformation. In other words, there is still work to do to complete this code, but you are welcome to adopt or reuse the code.
:::

# Get collection list

First, set up the tools required for this operation. The following cell imports libraries to work with CSV and JSON data, the Path tools for working with filepaths, and sets up the structures for importing and writing out data.

In [1]:
import csv
import json

# for working with local files
from pathlib import Path

DATA = Path('..', 'data')
OUT = Path('..', 'output')

# Transformation Part 1

At a high level, the transformation step involves the creation of code or another implementation workflow, which will search the item metadata files downloaded previously, extract the target fields identified in the MAP, then write that information to a CSV for ingest or loading into a new presentation platform.

## Developing the Automated Process

First, use the MAP to guide development of a script to accomplish the metadata transformation. One initial step is to identify all of the desired item metadata files (in JSON) gathered in the metadata extraction step. The process below begins by creating a "set list" of the files that you want to transform, called `list_of_item_metadata_files`, which are pulled from the `output` folder [where they were created in the harvesting step](/part03/apis-03-harvesting-item-metadata.ipynb).

:::{note} Reminder
This step builds on your terminal skills! (And builds on your understanding of regular expressions and shell navigation. Note, however, these are technically file path expansions, not actual regular expressions. The general idea of creating a pattern and asking the computer to respond with a list of results that meet your criteria, is similar.)
:::

In [3]:
metadata_file_path = OUT / Path('item-metadata')

print(metadata_file_path)

../output/item-metadata


The next cell uses the `glob` function, which allows the use of file path expanders
to look for patterns in file paths. In this case, the previous item metadata exraction
wrote files that had the pattern `item_metadata-[item-identifier].json`. 
So, to match any pattern for the `item-identifier` section, `glob` allows
the use of the `*` (asterisk) character to match any pattern:

In [4]:
file_count = 0

for file in metadata_file_path.glob('item_metadata-*.json'):
    file_count += 1
    print(file)
    
print('found',file_count)

../output/item-metadata/item_metadata-cph.3f05183.json
../output/item-metadata/item_metadata-g3851e.ct006252.json
../output/item-metadata/item_metadata-ppmsca.18016.json
../output/item-metadata/item_metadata-highsm.20497.json
../output/item-metadata/item_metadata-ppmsca.35590.json
../output/item-metadata/item_metadata-highsm.34640.json
../output/item-metadata/item_metadata-highsm.20336.json
../output/item-metadata/item_metadata-ppmsca.15412.json
../output/item-metadata/item_metadata-hhh.hi0135.photos.json
../output/item-metadata/item_metadata-det.4a17925.json
../output/item-metadata/item_metadata-ppmscd.00084.json
../output/item-metadata/item_metadata-hhh.ks0072.photos.json
../output/item-metadata/item_metadata-highsm.64003.json
../output/item-metadata/item_metadata-highsm.41101.json
../output/item-metadata/item_metadata-cph.3b43255.json
../output/item-metadata/item_metadata-hhh.me0057.photos.json
../output/item-metadata/item_metadata-highsm.36052.json
../output/item-metadata/item_meta

In [7]:
list_of_item_metadata_files = list() 
for file in metadata_file_path.glob('item_metadata-*.json'):
    list_of_item_metadata_files.append(file)

In [8]:
len(list_of_item_metadata_files)

30

To see if there are duplicate files identifiable by name, use a sort to quickly scan the list
and ensure that these are unique items.

In [ ]:
list_of_item_metadata_files.sort()

for file in list_of_item_metadata_files:
    print(file)

../output/item-metadata/item_metadata-cph.3b43255.json
../output/item-metadata/item_metadata-cph.3f05168.json
../output/item-metadata/item_metadata-cph.3f05183.json
../output/item-metadata/item_metadata-det.4a17925.json
../output/item-metadata/item_metadata-ds.06560.json
../output/item-metadata/item_metadata-fsa.8b32222.json
../output/item-metadata/item_metadata-fsa.8d24709.json
../output/item-metadata/item_metadata-g3851e.ct006252.json
../output/item-metadata/item_metadata-hhh.hi0135.photos.json
../output/item-metadata/item_metadata-hhh.ks0072.photos.json
../output/item-metadata/item_metadata-hhh.me0057.photos.json
../output/item-metadata/item_metadata-highsm.18384.json
../output/item-metadata/item_metadata-highsm.20336.json
../output/item-metadata/item_metadata-highsm.20483.json
../output/item-metadata/item_metadata-highsm.20497.json
../output/item-metadata/item_metadata-highsm.29207.json
../output/item-metadata/item_metadata-highsm.34640.json
../output/item-metadata/item_metadata-hi

To develop your data transformation, you need to understand how to retrieve the information that
the MAP requires for each item, then determine if it is necessary to modify this information for the load format. 
One way to do this is to explore one item as an exemplar for understanding how the information is structured.
How do you retrieve the information from the JSON structure? Where is the information you want? How is the information that you need formatted?

In [10]:
# try first with one file, can you open the json, can you see what elements are in the json?
with open(list_of_item_metadata_files[0], 'r', encoding='utf-8') as item:
    # what are we looking at?
    print('file:',list_of_item_metadata_files[0],'\n')
    
    # load the item data
    item_data = json.load(item)
    
    for element in item_data.keys():
        print(element,':',item_data[element])

file: ../output/item-metadata/item_metadata-cph.3b43255.json 

_version_ : 1731234817723334656
access_restricted : False
aka : ['https://www.loc.gov/pictures/item/89710983/', 'http://www.loc.gov/item/89710983/', 'http://www.loc.gov/pictures/item/89710983/', 'https://www.loc.gov/pictures/collection/finepr/item/89710983/', 'http://www.loc.gov/pictures/collection/finepr/item/89710983/', 'http://www.loc.gov/resource/cph.3b43255/', 'http://lccn.loc.gov/89710983', 'https://hdl.loc.gov/loc.pnp/cph.3b43255']
call_number : FP - XX - D9922, no. 12 (A size) [P&P]
campaigns : []
contributor_names : ['Dwight, Mabel, 1876-1955, artist']
contributors : [{'dwight, mabel': 'https://www.loc.gov/search/?fa=contributor:dwight,+mabel&fo=json'}]
control_number : 
created : 2021-08-05T17:15:08Z
created_published : ['[1939]']
created_published_date : [1939]
date : 1939-01-01
dates : [{'1939': 'https://www.loc.gov/search/?dates=1939/1939&fo=json'}]
description : ['1 print : lithograph. | Group of people readin

Look around in the dictionary a bit more:

In [11]:
item_data.keys()

dict_keys(['_version_', 'access_restricted', 'aka', 'call_number', 'campaigns', 'contributor_names', 'contributors', 'control_number', 'created', 'created_published', 'created_published_date', 'date', 'dates', 'description', 'digital_id', 'digitized', 'display_offsite', 'extract_timestamp', 'extract_urls', 'format', 'format_headings', 'genre', 'group', 'hassegments', 'id', 'image_url', 'index', 'item', 'language', 'languages', 'library_of_congress_control_number', 'link', 'marc', 'medium', 'medium_brief', 'mime_type', 'modified', 'notes', 'number', 'number_former_id', 'number_lccn', 'number_source_modified', 'online_format', 'original_format', 'other_control_numbers', 'other_formats', 'partof', 'related', 'reproduction_number', 'reproductions', 'resource_links', 'resources', 'rights', 'rights_advisory', 'rights_information', 'score', 'shard', 'shelf_id', 'site', 'sort_date', 'source_created', 'source_modified', 'stmt_of_responsibility', 'subject', 'subject_headings', 'subjects', 'summa

To develop the metadata transformation, the script needs clear instructions about
how to extract the elements identified in the MAP table. For example, which date fields do you want and where are they? Where will you find the format information?

In [14]:
# what is the title?
print('title:',item_data['title'])
# can you get the date?
print('\ndate:',item_data['date'], type(item_data['date']))
# can you get the format?
print('\nformat:',item_data['format'][0], type(item_data['format']))

title: Silence / Mabel Dwight.

date: 1939-01-01 <class 'str'>

format: {'photo, print, drawing': 'https://www.loc.gov/search/?fa=original_format:photo,+print,+drawing&fo=json'} <class 'list'>


## Testing the Process

Test the extract process with at least one example. As you have larger sets of items, this testing process must become more complex to allow for multiple cases. Testing will thus become more complicated as you go along since it is possible that metadata attributes in the JSON files will be different for different types of items.
But start with some basics and build up from there. 

For a first pass, look out for these items, and find where in the JSON you can locate them:

- 'item_id'
- 'title'
- 'date' 
- 'source_url'
- 'phys_format'
- 'dig_format'
- 'rights'

:::{hint} Use a JSON viewer 
While it's possible to explore the data with a script-based approach as shown below, it may be quicker or more efficient for a person to examine the JSON structure directly. To do this, use an extension in VSCode, the viewer in JupyterLab, or a browser extension that will display the JSON structure. The block below uses data from the item with LC resource identifier `cph.3b41963`.
:::

You may need to use try/except patterns to create workarounds for cases where some items may not have exactly the same attributes that you've identified in your test cases.

In [15]:
# set up the containers to create the csv of all the item fields
# file for csv to read out
collection_info_csv = OUT / 'collection_items_data.csv'

# set up a list for the columns in your csv; 
# your goal should be to automate this, but . . . 
# it works for demonstration as you set up the crosswalk
headers = ['source_file', 'item_id', 'title', 'date', 'source_url', 'phys_format', 'dig_format', 'rights']

# try first with one file
with open(list_of_item_metadata_files[0], 'r', encoding='utf-8') as data:
    # load the item data
    item_data = json.load(data)
    
    # extract the data you want
    # for checking purposes, add in the source of the info
    source_file = str(file)
    # make sure there's some unique and stable identifier
    try:
        item_id = item_data['library_of_congress_control_number']
    except:
        item_id = item_data['url'].split('/')[-2]
    title = item_data['title']
    date = item_data['date']
    source_url = item_data['url']
    try:
        phys_format = item_data['format'][0]
    except:
        phys_format = 'Not found'
    try:
        dig_format = item_data['online_format'][0]
    except:
        dig_format = 'Not found'
    mime_type = item_data['mime_type']
    try:
        rights = item_data['rights_information']
    except:
        rights = 'Undetermined'


    # dictionary for the rows
    row_dict = dict()
    
    # look for the item metadata, assign it to the dictionary; 
    # start with some basic elements likely (already enumerated in the headers list) :
    # source file
    row_dict['source_file'] = source_file
    # identifier
    row_dict['item_id'] = item_id
    # title
    row_dict['title'] = title
    # date
    row_dict['date'] = date
    # link
    row_dict['source_url'] = source_url
    # format
    row_dict['phys_format'] = phys_format
    # digital format
    row_dict['dig_format'] = dig_format
    #rights
    row_dict['rights'] = rights 
    print('created row dictionary:',row_dict)

    # write to the csv
    with open(collection_info_csv, 'w', encoding='utf-8') as fout:
        writer = csv.DictWriter(fout, fieldnames=headers)
        writer.writeheader()
        writer.writerow(row_dict)
        print('wrote',collection_info_csv)

created row dictionary: {'source_file': '../output/item-metadata/item_metadata-ppmscd.00084.json', 'item_id': '89710983', 'title': 'Silence / Mabel Dwight.', 'date': '1939-01-01', 'source_url': 'https://www.loc.gov/item/89710983/', 'phys_format': {'photo, print, drawing': 'https://www.loc.gov/search/?fa=original_format:photo,+print,+drawing&fo=json'}, 'dig_format': 'image', 'rights': 'No known restrictions on publication. For more information, see Federal Art Project (FAP) Prints  - Rights and Restrictions Information http://hdl.loc.gov/loc.pnp/res.750.fap'}
wrote ../output/collection_items_data.csv


You're now developing the structure of the CSV file that will import items into your Omeka S site. The CSV import module supports the loading of item files via a URL. This provides the location of a file (in this case, an image), which Omeka will copy into its database and attach to your item. This means that it isn't necessary to upload individual files after or during metadata creation. 

To allow this, you need to find a direct url to a good image file for the item. There are multiple options, and the code below demonstrates looking for the url to a medium-sized image of an item:

In [19]:
collection_info_csv = OUT / 'collection_items_data.csv'

# set up a list for the columns in your csv; in future, this should be more automated but this works for now as you set up the crosswalk
headers = ['source_file', 'item_id', 'title', 'date', 'source_url', 'phys_format', 'dig_format', 'rights']

# try first with one file
with open(list_of_item_metadata_files[0], 'r', encoding='utf-8') as data:
    # load the item data
    item_data = json.load(data)
    
    print(item_data['image_url'][2])

https://tile.loc.gov/storage-services/service/pnp/cph/3b40000/3b43000/3b43200/3b43255r.jpg#h=422&w=640


# Transformation Part 2

The goal of this step is to create data for the final load step. In this case, a CSV file, which will be possible to import into an Omeka S site using the CSV Import module. It may seem like it's taken a long time to get to this point... but remember, when this works you will be importing around 60 items into the site at one time, so if you can get all of this to work for an even larger set of materials, you will be saving quite a lot of time in the future when you need to import items. Even if you were to collect the items piecemeal, which would need a different workflow than illustrated here, you can accomplish similar goals by recording metadata for each item consistently and in a spreadsheet, which you can then use to import the items in batch.

So now that your transformation script is tested, the goal is to extend this to the whole set by looping through each of the desired JSON files:

In [20]:
# for purposes of demonstration, use this block to make sure there isn't already a list file:

items_data_file = OUT / 'collection_items_data.csv'

if items_data_file.is_file():
    items_data_file.unlink()
    print('removed',items_data_file)

# clear row_dict
row_dict = ()

removed ../output/collection_items_data.csv


In [21]:
from datetime import date

date_string_for_today = date.today().strftime('%Y-%m-%d') # see https://docs.python.org/3/library/datetime.html#strftime-strptime-behavior

print(date_string_for_today)

2026-11-13


In [22]:
# set up the containers to create the csv & counters 
# file for csv to read out
collection_info_csv = OUT / 'collection_items_data.csv'
file_count = 0
items_written = 0
error_count = 0

# add in a couple of extras for Omeka, including item type and date uploaded

# set up a list for the columns in your csv; in future, this should be more automated but this works for now as you set up the crosswalk
headers = ['item_type', 'date_uploaded', 'source_file', 'item_id', 'title', 'date', 'source_url', 'phys_format', 'dig_format', 'rights', 'image_url']

# now, adapt the previous loop to open each file:
for file in list_of_item_metadata_files:
    file_count += 1
    print('opening',file)
    with open(file, 'r', encoding='utf-8') as item:
        # load the item data
        try:
            item_data = json.load(item)
        except:
            print('error loading',file)
            error_count += 1
            continue

        # extract/name the data you want
        # item type
        item_type = 'Item'
        # date uplaoded
        date_uploaded = date_string_for_today
        # for checking purposes, add in the source of the info
        source_file = str(file)
        # make sure there's some unique and stable identifier
        try:
            item_id = item_data['library_of_congress_control_number']
        except:
            item_id = item_data['url'].split('/')[-2]
        title = item_data['title']
        try:
            date = item_data['date']
        except:
            date = 'Not found'
        source_url = item_data['url']
        try:
            phys_format = item_data['format'][0]
        except:
            phys_format = 'Not found'
        try:
            dig_format = item_data['online_format'][0]
        except:
            dig_format = 'Not found'
        mime_type = item_data['mime_type']
        try:
            rights = item_data['rights_information']
        except:
            rights = 'Undetermined'
        try:
            image_url = item_data['image_url'][3]
        except:
            image_url = 'Did not identify a URL.'

        # dictionary for the rows
        row_dict = dict()

        # look for the item metadata, assign it to the dictionary; 
        # start with some basic elements likely (already enumerated in the headers list) :
        # item type
        row_dict['item_type'] = item_type
        # date uploaded
        row_dict['date_uploaded'] = date_uploaded
        # source filename
        row_dict['source_file'] = source_file
        # identifier
        row_dict['item_id'] = item_id
        # title
        row_dict['title'] = title
        # date
        row_dict['date'] = date
        # link
        row_dict['source_url'] = source_url
        # format
        row_dict['phys_format'] = phys_format
        # digital format
        row_dict['dig_format'] = dig_format.capitalize()
        #rights
        row_dict['rights'] = rights
        #image
        row_dict['image_url'] = image_url

        # write to the csv
        with open(collection_info_csv, 'a', encoding='utf-8') as fout:
            writer = csv.DictWriter(fout, fieldnames=headers)
            if items_written == 0:
                writer.writeheader()
            writer.writerow(row_dict)
            items_written += 1
            print('adding',item_id)

print('\n\n--- LOG ---')
print('wrote',collection_info_csv)
print('with',items_written,'items')
print(error_count,'errors (info not written)')

opening ../output/item-metadata/item_metadata-cph.3b43255.json
adding 89710983
opening ../output/item-metadata/item_metadata-cph.3f05168.json
adding 98508385
opening ../output/item-metadata/item_metadata-cph.3f05183.json
adding 98508155
opening ../output/item-metadata/item_metadata-det.4a17925.json
adding 2016809661
opening ../output/item-metadata/item_metadata-ds.06560.json
adding 2014647618
opening ../output/item-metadata/item_metadata-fsa.8b32222.json
adding 2017770391
opening ../output/item-metadata/item_metadata-fsa.8d24709.json
adding 2017843202
opening ../output/item-metadata/item_metadata-g3851e.ct006252.json
adding 87694100
opening ../output/item-metadata/item_metadata-hhh.hi0135.photos.json
adding hi0135
opening ../output/item-metadata/item_metadata-hhh.ks0072.photos.json
adding ks0072
opening ../output/item-metadata/item_metadata-hhh.me0057.photos.json
adding me0057
opening ../output/item-metadata/item_metadata-highsm.18384.json
adding 2013646466
opening ../output/item-metad

Now, you should have a well-formed, complete CSV file at `data/collection_items_data.csv`. This file should include all of the information for the 59 items that were fully described in the extract step. 